# Handling Class Imbalance

Apply class weighting to improve fraud detection performance.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HandleImbalance") \
    .master("local[*]") \
    .getOrCreate()

train_df = spark.read.parquet("../data/processed/train")
test_df = spark.read.parquet("../data/processed/test")

print("Data loaded")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 17:07:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Data loaded


### 2. Compute class weights

In [2]:
from pyspark.sql.functions import col

fraud_count = train_df.filter(col("label") == 1).count()
legit_count = train_df.filter(col("label") == 0).count()

total = fraud_count + legit_count

fraud_weight = total / (2 * fraud_count)
legit_weight = total / (2 * legit_count)

print("Fraud weight:", fraud_weight)
print("Legit weight:", legit_weight)

Fraud weight: 386.4431110435971
Legit weight: 0.5006477638616842


### 3. Add weight column

In [3]:
from pyspark.sql.functions import when, col

train_weighted = train_df.withColumn(
    "weight",
    when(col("label") == 1, fraud_weight).otherwise(legit_weight)
)

### 4. Train Weighted Model

In [4]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=20,
    maxDepth=5,
    seed=42
)

pipeline = Pipeline(stages=[rf])
model_weighted = pipeline.fit(train_weighted)

print("Weighted model trained")

26/05/13 17:07:40 WARN MemoryStore: Not enough space to cache rdd_46_3 in memory! (computed 211.0 MiB so far)
26/05/13 17:07:40 WARN MemoryStore: Not enough space to cache rdd_46_10 in memory! (computed 211.0 MiB so far)
26/05/13 17:07:40 WARN BlockManager: Persisting block rdd_46_3 to disk instead.
26/05/13 17:07:40 WARN BlockManager: Persisting block rdd_46_10 to disk instead.
26/05/13 17:07:46 WARN MemoryStore: Not enough space to cache rdd_46_10 in memory! (computed 211.0 MiB so far)
26/05/13 17:07:46 WARN MemoryStore: Not enough space to cache rdd_46_3 in memory! (computed 211.0 MiB so far)
26/05/13 17:07:50 WARN MemoryStore: Not enough space to cache rdd_46_3 in memory! (computed 211.0 MiB so far)
26/05/13 17:07:50 WARN MemoryStore: Not enough space to cache rdd_46_10 in memory! (computed 211.0 MiB so far)
26/05/13 17:07:54 WARN MemoryStore: Not enough space to cache rdd_46_10 in memory! (computed 211.0 MiB so far)
26/05/13 17:07:54 WARN MemoryStore: Not enough space to cache rdd

Weighted model trained


### 5. Generate Predictions

In [5]:
predictions = model_weighted.transform(test_df)

predictions.select("label", "prediction").show(5)

[Stage 27:===========================================>              (3 + 1) / 4]

+-----+----------+
|label|prediction|
+-----+----------+
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
+-----+----------+
only showing top 5 rows



### 6. Confusion matrix

In [6]:
predictions.groupBy("label", "prediction").count().show()

[Stage 28:================================================>       (13 + 2) / 15]

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0|1273079|
|    0|       1.0|      1|
|    1|       1.0|   1621|
|    1|       0.0|      9|
+-----+----------+-------+



### 7. Fraud recall

In [7]:
tp = predictions.filter((col("label") == 1) & (col("prediction") == 1)).count()
fn = predictions.filter((col("label") == 1) & (col("prediction") == 0)).count()

fraud_recall = tp / (tp + fn)

print("Fraud Recall (weighted):", fraud_recall)

Fraud Recall (weighted): 0.994478527607362


### 8. Fraud precision

In [8]:
fp = predictions.filter((col("label") == 0) & (col("prediction") == 1)).count()

fraud_precision = tp / (tp + fp)

print("Fraud Precision (weighted):", fraud_precision)

[Stage 37:================================================>       (13 + 2) / 15]

Fraud Precision (weighted): 0.9993834771886559


## Results

Class weighting was applied to address strong imbalance.

### Observations:
- fraud recall slightly improved compared to baseline
- model shows better sensitivity to fraud cases

### Trade-off:
- no significant increase in false positives observed

### Conclusion:
Handling imbalance improves model focus on fraud detection.